In [1]:
import os, sys, shutil, subprocess, zipfile, random, io, warnings
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm
import requests

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
ON_KAGGLE  = os.path.exists("/kaggle")
BASE_DIR   = Path("/kaggle/working") if ON_KAGGLE else Path("./data")
DATA_DIR   = BASE_DIR / "datasets"
CKPT_DIR   = BASE_DIR / "checkpoints"
TU_DIR     = DATA_DIR / "tu_berlin"
SK_DIR     = DATA_DIR / "sketchy"
QD_DIR     = DATA_DIR / "quickdraw"
SKETCH_DIR = DATA_DIR / "sketch_all"
PHOTO_DIR  = DATA_DIR / "photo_all"

for d in [DATA_DIR, CKPT_DIR, TU_DIR, SK_DIR, QD_DIR, SKETCH_DIR, PHOTO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMG_SIZE     = 256
BATCH_SIZE   = 4
NUM_EPOCHS   = 50
LR           = 2e-4
BETAS        = (0.5, 0.999)
LAMBDA_CYCLE = 10.0
LAMBDA_IDT   = 5.0
N_RES        = 9
NGF = NDF    = 64
POOL_SIZE    = 50
NUM_WORKERS  = min(4, os.cpu_count() or 1)

QD_BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap"
QD_CATS = [
    "airplane", "bicycle", "bird", "boat", "car",
    "cat", "chair", "clock", "dog", "elephant",
    "fish", "flower", "guitar", "horse", "house",
    "shoe", "sun", "tree", "umbrella", "face"
]
QD_MAX_PER = 600

In [3]:
subprocess.run("pip install -q huggingface_hub datasets", shell=True)

CompletedProcess(args='pip install -q huggingface_hub datasets', returncode=0)

In [ ]:
def download_tu_berlin(dst: Path) -> int:
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
    existing = [p for p in dst.rglob("*") if p.suffix.lower() in img_exts]
    if len(existing) >= 100:
        return len(existing)

    count = 0
    try:
        from datasets import load_dataset
        ds = load_dataset("sdiaeyu6n/tu-berlin", trust_remote_code=True)
        for split_name, split in ds.items():
            img_col = next((c for c in split.column_names
                            if any(k in c.lower() for k in ['image','img','sketch','pixel'])), split.column_names[0])
            for sample in tqdm(split, desc=f"TU-Berlin {split_name}"):
                val = sample[img_col]
                try:
                    if isinstance(val, Image.Image):
                        img = val.convert('RGB')
                    elif isinstance(val, dict) and 'bytes' in val:
                        img = Image.open(io.BytesIO(val['bytes'])).convert('RGB')
                    elif isinstance(val, bytes):
                        img = Image.open(io.BytesIO(val)).convert('RGB')
                    elif isinstance(val, np.ndarray):
                        img = Image.fromarray(val.astype(np.uint8)).convert('RGB')
                    else:
                        continue
                    img.save(dst / f"tu_{count:07d}.png")
                    count += 1
                except Exception:
                    pass
        if count > 0:
            return count
    except Exception:
        pass

    try:
        from huggingface_hub import snapshot_download
        repo_path = snapshot_download(repo_id="sdiaeyu6n/tu-berlin", repo_type="dataset",
                                      local_dir=str(dst / "_raw"))
        for p in Path(repo_path).rglob("*"):
            if p.suffix.lower() in img_exts and p.is_file():
                try:
                    Image.open(p).convert('RGB').save(dst / f"tu_{count:07d}{p.suffix}")
                    count += 1
                except Exception:
                    pass
        shutil.rmtree(dst / "_raw", ignore_errors=True)
    except Exception:
        pass

    return count

download_tu_berlin(TU_DIR)

In [ ]:
def run_kaggle_download(dataset_id: str, dst: Path) -> bool:
    res = subprocess.run(f"kaggle datasets download -d {dataset_id} -p {dst} --unzip",
                         shell=True, capture_output=True, text=True, timeout=600)
    if res.returncode != 0:
        return False
    for zf in dst.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zf) as z: z.extractall(zf.parent)
            zf.unlink()
        except Exception:
            pass
    return True


def find_sketch_photo_dirs(root: Path):
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.gif'}
    sk_dirs, ph_dirs = [], []

    def has_images(d):
        return any(p.suffix.lower() in img_exts for p in d.iterdir() if p.is_file())

    for dirpath, _, _ in os.walk(root):
        p = Path(dirpath)
        if not p.is_dir() or not any(p.iterdir()): continue
        try:
            if not has_images(p): continue
        except Exception:
            continue
        nm = p.name.lower()
        parent_nm = p.parent.name.lower()
        if any(k in nm for k in ['sketch', 'drawing', 'svg', 'line']):
            sk_dirs.append(p)
        elif any(k in nm for k in ['photo', 'image', 'img', 'pic', 'rgb', 'ref', 'real']):
            ph_dirs.append(p)
        elif any(k in parent_nm for k in ['sketch', 'drawing']):
            sk_dirs.append(p)
        elif any(k in parent_nm for k in ['photo', 'image', 'img']):
            ph_dirs.append(p)
    return sk_dirs, ph_dirs


def download_sketchy(sk_raw: Path, sketch_dst: Path, photo_dst: Path):
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

    for did in ["prasunroy/sketchy", "datasciencemaster/sketchy", "sharanyasundar/sketchy-dataset"]:
        ok = run_kaggle_download(did, sk_raw)
        if ok and sum(1 for p in sk_raw.rglob("*") if p.suffix.lower() in img_exts) >= 50:
            break

    sk_dirs, ph_dirs = find_sketch_photo_dirs(sk_raw)
    n_sk = n_ph = 0

    for d in tqdm(sk_dirs, desc="Sketchy sketches"):
        for p in d.glob("*"):
            if p.is_file() and p.suffix.lower() in img_exts:
                try:
                    Image.open(p).convert('RGB').save(sketch_dst / f"sk_{n_sk:07d}.png")
                    n_sk += 1
                except Exception:
                    pass

    for d in tqdm(ph_dirs, desc="Sketchy photos"):
        for p in d.glob("*"):
            if p.is_file() and p.suffix.lower() in img_exts:
                try:
                    Image.open(p).convert('RGB').save(photo_dst / f"skph_{n_ph:07d}.png")
                    n_ph += 1
                except Exception:
                    pass

download_sketchy(SK_DIR, SKETCH_DIR, PHOTO_DIR)